In [14]:
import numpy as np
import cv2
import os

print("🎯 ДЗ: ВЕЙВЛЕТ-ПРЕОБРАЗОВАНИЕ ХААРА И СЖАТИЕ ИЗОБРАЖЕНИЙ")
print("=" * 70)
print("✅ Библиотеки загружены!")

🎯 ДЗ: ВЕЙВЛЕТ-ПРЕОБРАЗОВАНИЕ ХААРА И СЖАТИЕ ИЗОБРАЖЕНИЙ
✅ Библиотеки загружены!


# 🎯 ДЗ: ВЕЙВЛЕТ-ПРЕОБРАЗОВАНИЕ ХААРА И СЖАТИЕ ИЗОБРАЖЕНИЙ

## 1. 📷 ЗАГРУЗКА И СОХРАНЕНИЕ МОНОХРОМНОГО ИЗОБРАЖЕНИЯ

In [15]:
# Загрузка изображения
image = cv2.imread("image.jpg", cv2.IMREAD_GRAYSCALE)
if image is None:
    print("❌ ОШИБКА: Файл 'image.jpg' не найден!")
    exit()

print("✅ Изображение успешно загружено!")
print(f"   Размер: {image.shape}")
print(f"   Тип данных: {image.dtype}")

# Сохранение в текстовый файл
np.savetxt('original.txt', image, fmt='%d')
print("✅ Исходное изображение сохранено в файл 'original.txt'")

# Сохранение монохромного изображения в отдельный файл
np.savetxt('monochrome.txt', image, fmt='%d')
print("✅ Монохромное изображение сохранено в файл 'monochrome.txt'")

✅ Изображение успешно загружено!
   Размер: (1080, 1080)
   Тип данных: uint8
✅ Исходное изображение сохранено в файл 'original.txt'
✅ Монохромное изображение сохранено в файл 'monochrome.txt'


## 2. 🔄 РЕАЛИЗАЦИЯ ВЕЙВЛЕТ-ПРЕОБРАЗОВАНИЯ ХААРА

In [16]:
def haar_wavelet_transform(image):
    """
    Двумерное вейвлет-преобразование Хаара
    Возвращает компоненты: LL, LH, HL, HH
    """
    # Приведение к четным размерам
    h, w = image.shape
    h_even = h - h % 2
    w_even = w - w % 2
    img = image[:h_even, :w_even].astype(np.float32)
    
    print(f"✅ Размер после приведения к четным размерам: {img.shape}")
    
    half_h, half_w = h_even // 2, w_even // 2
    transformed = np.zeros_like(img)
    
    # Преобразование по строкам
    for i in range(h_even):
        row = img[i, :]
        # Вычисление средних и разностей
        avg = (row[0::2] + row[1::2]) * 0.5
        diff = (row[0::2] - row[1::2]) * 0.5
        transformed[i, :half_w] = avg
        transformed[i, half_w:] = diff
    
    # Преобразование по столбцам
    result = np.zeros_like(transformed)
    for j in range(w_even):
        col = transformed[:, j]
        # Вычисление средних и разностей
        avg = (col[0::2] + col[1::2]) * 0.5
        diff = (col[0::2] - col[1::2]) * 0.5
        result[:half_h, j] = avg
        result[half_h:, j] = diff
    
    # Разделение на компоненты
    LL = result[:half_h, :half_w]          # Низкие частоты (приближение)
    LH = result[:half_h, half_w:]          # Горизонтальные детали
    HL = result[half_h:, :half_w]          # Вертикальные детали  
    HH = result[half_h:, half_w:]          # Диагональные детали
    
    print("✅ Вейвлет-преобразование Хаара выполнено")
    print(f"   Размеры компонентов: LL{LL.shape}, LH{LH.shape}, HL{HL.shape}, HH{HH.shape}")
    
    return LL, LH, HL, HH

# Применение преобразования Хаара
LL, LH, HL, HH = haar_wavelet_transform(image)

✅ Размер после приведения к четным размерам: (1080, 1080)
✅ Вейвлет-преобразование Хаара выполнено
   Размеры компонентов: LL(540, 540), LH(540, 540), HL(540, 540), HH(540, 540)


## 3. 📊 КВАНТОВАНИЕ ВЫСОКОЧАСТОТНЫХ КОМПОНЕНТ

In [17]:
def quantize_high_freq(data, levels=4):
    """
    Квантование высокочастотных компонентов
    levels - количество уровней квантования
    """
    dmin, dmax = np.min(data), np.max(data)
    
    if dmax == dmin:
        # Все значения одинаковые
        quantized = np.zeros_like(data, dtype=int)
        return quantized, dmin, 1.0
    
    # Линейное квантование
    step = (dmax - dmin) / (levels - 1)
    quantized = np.round((data - dmin) / step).astype(int)
    quantized = np.clip(quantized, 0, levels - 1)
    
    print(f"✅ Квантование выполнено: диапазон [{dmin:.3f}, {dmax:.3f}], шаг {step:.3f}")
    print(f"   Уникальные значения: {np.unique(quantized)}")
    
    return quantized, dmin, step

# Квантование высокочастотных компонентов
print("🔹 КВАНТОВАНИЕ LH (горизонтальные детали)")
LH_q, lh_min, lh_step = quantize_high_freq(LH, 4)

print("\n🔹 КВАНТОВАНИЕ HL (вертикальные детали)")
HL_q, hl_min, hl_step = quantize_high_freq(HL, 4)

print("\n🔹 КВАНТОВАНИЕ HH (диагональные детали)")
HH_q, hh_min, hh_step = quantize_high_freq(HH, 4)

🔹 КВАНТОВАНИЕ LH (горизонтальные детали)
✅ Квантование выполнено: диапазон [-53.750, 44.750], шаг 32.833
   Уникальные значения: [0 1 2 3]

🔹 КВАНТОВАНИЕ HL (вертикальные детали)
✅ Квантование выполнено: диапазон [-45.000, 45.000], шаг 30.000
   Уникальные значения: [0 1 2 3]

🔹 КВАНТОВАНИЕ HH (диагональные детали)
✅ Квантование выполнено: диапазон [-32.250, 28.750], шаг 20.333
   Уникальные значения: [0 1 2 3]


## 4. 💾 СЖАТИЕ ДЛИН СЕРИЙ И СОХРАНЕНИЕ РЕЗУЛЬТАТОВ

In [18]:
def run_length_encode(array):
    """
    Кодирование длин серий (RLE)
    Возвращает список пар (значение, количество повторений)
    """
    flat = array.flatten()
    if len(flat) == 0:
        return []
    
    encoded = []
    prev = flat[0]
    count = 1
    
    for value in flat[1:]:
        if value == prev:
            count += 1
        else:
            encoded.append((prev, count))
            prev = value
            count = 1
    
    encoded.append((prev, count))
    
    original_size = len(flat)
    compressed_size = len(encoded) * 2  # Каждая пара - 2 числа
    compression_ratio = original_size / compressed_size if compressed_size > 0 else 1
    
    print(f"   RLE: {original_size} → {compressed_size} элементов (коэф. {compression_ratio:.2f})")
    
    return encoded

# Применение RLE к квантованным компонентам
print("🔹 СЖАТИЕ LH:")
LH_rle = run_length_encode(LH_q)

print("\n🔹 СЖАТИЕ HL:")
HL_rle = run_length_encode(HL_q)

print("\n🔹 СЖАТИЕ HH:")
HH_rle = run_length_encode(HH_q)

def save_compressed_data(filename, LL, LH_rle, HL_rle, HH_rle):
    """
    Сохранение сжатых данных в текстовый файл
    Формат: LL, затем LH, HL, HH в формате RLE
    """
    with open(filename, 'w') as f:
        # Заголовок с размерами
        f.write(f"# Вейвлет-преобразование Хаара с RLE сжатием\n")
        f.write(f"# Размер LL: {LL.shape[0]} {LL.shape[1]}\n")
        f.write(f"# Количество серий: LH={len(LH_rle)}, HL={len(HL_rle)}, HH={len(HH_rle)}\n\n")
        
        # Сохранение LL компонента
        f.write("LL_COMPONENT\n")
        np.savetxt(f, LL, fmt='%.6f')
        f.write("\n")
        
        # Сохранение LH компонента в формате RLE
        f.write("LH_COMPONENT_RLE\n")
        for value, count in LH_rle:
            f.write(f"{value} {count}\n")
        f.write("\n")
        
        # Сохранение HL компонента в формате RLE
        f.write("HL_COMPONENT_RLE\n")
        for value, count in HL_rle:
            f.write(f"{value} {count}\n")
        f.write("\n")
        
        # Сохранение HH компонента в формате RLE
        f.write("HH_COMPONENT_RLE\n")
        for value, count in HH_rle:
            f.write(f"{value} {count}\n")

# Сохранение сжатых данных
save_compressed_data('haar_compressed.txt', LL, LH_rle, HL_rle, HH_rle)
print("✅ Сжатые данные сохранены в файл 'haar_compressed.txt'")

🔹 СЖАТИЕ LH:
   RLE: 291600 → 153336 элементов (коэф. 1.90)

🔹 СЖАТИЕ HL:
   RLE: 291600 → 267608 элементов (коэф. 1.09)

🔹 СЖАТИЕ HH:
   RLE: 291600 → 256262 элементов (коэф. 1.14)
✅ Сжатые данные сохранены в файл 'haar_compressed.txt'


## 5. 📈 СРАВНЕНИЕ ОБЪЕМОВ ПАМЯТИ

In [19]:
# Вычисление размеров файлов
original_size = os.path.getsize('original.txt')
monochrome_size = os.path.getsize('monochrome.txt')
compressed_size = os.path.getsize('haar_compressed.txt')

# Размер в памяти (попиксельное хранение)
memory_size = image.nbytes

print("📊 СРАВНЕНИЕ РАЗМЕРОВ:")
print(f"   Исходное изображение в памяти: {memory_size:,} байт")
print(f"   Файл original.txt: {original_size:,} байт")
print(f"   Файл monochrome.txt: {monochrome_size:,} байт")
print(f"   Файл haar_compressed.txt: {compressed_size:,} байт")

# Вычисление коэффициентов сжатия
compression_ratio_memory = memory_size / compressed_size
compression_ratio_file = original_size / compressed_size
size_reduction = ((original_size - compressed_size) / original_size) * 100

print(f"\n🎯 РЕЗУЛЬТАТЫ СЖАТИЯ:")
print(f"   Коэффициент сжатия (память): {compression_ratio_memory:.2f}:1")
print(f"   Коэффициент сжатия (файлы): {compression_ratio_file:.2f}:1")
print(f"   Экономия места: {size_reduction:.1f}%")

# Анализ распределения данных
print(f"\n🔍 АНАЛИЗ РАСПРЕДЕЛЕНИЯ ДАННЫХ:")
print(f"   LL компонент: {LL.shape[0]}×{LL.shape[1]} = {LL.size} элементов")
print(f"   LH компонент: {len(LH_rle)} серий из {sum(count for _, count in LH_rle)} элементов")
print(f"   HL компонент: {len(HL_rle)} серий из {sum(count for _, count in HL_rle)} элементов")
print(f"   HH компонент: {len(HH_rle)} серий из {sum(count for _, count in HH_rle)} элементов")

📊 СРАВНЕНИЕ РАЗМЕРОВ:
   Исходное изображение в памяти: 1,166,400 байт
   Файл original.txt: 3,733,491 байт
   Файл monochrome.txt: 3,733,491 байт
   Файл haar_compressed.txt: 4,678,777 байт

🎯 РЕЗУЛЬТАТЫ СЖАТИЯ:
   Коэффициент сжатия (память): 0.25:1
   Коэффициент сжатия (файлы): 0.80:1
   Экономия места: -25.3%

🔍 АНАЛИЗ РАСПРЕДЕЛЕНИЯ ДАННЫХ:
   LL компонент: 540×540 = 291600 элементов
   LH компонент: 76668 серий из 291600 элементов
   HL компонент: 133804 серий из 291600 элементов
   HH компонент: 128131 серий из 291600 элементов


## Выполненные задания:
- ✅ 1. Сохранение монохромного изображения в текстовый файл
- ✅ 2. Реализация алгоритма вейвлет-преобразования Хаара
- ✅ 3. Квантование высокочастотных компонент (4 уровня)
- ✅ 4. Сжатие длин серий и сохранение в файл
- ✅ 5. Сравнение объемов памяти до и после сжатия

## Созданные файлы:
- `original.txt` - исходное изображение
- `monochrome.txt` - монохромное изображение
- `haar_compressed.txt` - сжатые вейвлет-коэффициенты

## Итоговый результат:
- Коэффициент сжатия: вычисляется автоматически
- Экономия памяти: вычисляется автоматически